# FER-YOLO-Mamba — Kaggle Training Notebook

### Before you run
1. Add **both datasets** via the right panel → **+ Add data**
   - Your **code dataset** (`nets/`, `utils/`, `model_data/`, `train_kaggle.py`, `convert_raf_to_yolo.py`)
   - Your **RAF-DB dataset** (`basic-20260621T140957Z-3-001/`)
2. Enable **GPU**: Notebook Settings → Accelerator → **GPU T4 x1**
3. Run all cells top to bottom

---

In [ ]:
# ================================================================
# CELL 1 — ENVIRONMENT & GPU CHECK
# ================================================================
import os, sys, torch

WORK_DIR = '/kaggle/working'

assert os.path.exists('/kaggle/input'), 'This notebook must run on Kaggle.'

assert torch.cuda.is_available(), (
    'No GPU detected!\n'
    'Fix: Notebook Settings → Accelerator → GPU T4 x1, then re-run.'
)

p = torch.cuda.get_device_properties(0)
print('=' * 50)
print(f'  GPU     : {p.name}')
print(f'  VRAM    : {p.total_memory / 1e9:.1f} GB')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.version.cuda}')
print('=' * 50)

In [ ]:
# ================================================================
# CELL 2 — INSTALL DEPENDENCIES
# causal-conv1d is intentionally skipped — this repo only needs
# selective_scan_cuda which is compiled by mamba-ssm itself.
# ================================================================
import subprocess

# Help the compiler find CUDA headers
for _p in ['/usr/local/cuda', '/usr/local/cuda-12', '/usr/local/cuda-12.9']:
    if os.path.isdir(_p):
        os.environ['CUDA_HOME'] = _p
        os.environ['PATH'] = os.path.join(_p, 'bin') + ':' + os.environ.get('PATH', '')
        print(f'CUDA_HOME : {_p}')
        break

os.environ['MAX_JOBS'] = '4'

def _pip(pkg, flags=None):
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install'] + (flags or []) + [pkg],
        capture_output=True, text=True
    )
    return r.returncode == 0, r.stdout + '\n' + r.stderr

def _require(pkg, label):
    print(f'  {label}...', end=' ', flush=True)
    ok, log = _pip(pkg)
    if not ok:
        print('FAILED')
        for line in log.strip().splitlines()[-20:]: print('    ' + line)
        raise RuntimeError(f'pip install failed: {pkg}')
    print('OK')

print('Installing dependencies...\n')
_require('packaging',            '[1/3] packaging')
_require('einops',               '      einops')
_require('timm>=0.6.13,<0.9.0', '[2/3] timm')

# mamba-ssm: --no-deps skips causal-conv1d (which fails on Py3.12 + CUDA 12.x)
print('  [3/3] mamba-ssm (compiling CUDA kernel ~10 min)...', end=' ', flush=True)
_mamba_ok = False
for _flags in [['--no-deps'], ['--no-deps', '--no-build-isolation'], []]:
    ok, log = _pip('mamba-ssm>=2.0.3', _flags)
    if ok:
        print('OK')
        _mamba_ok = True
        break

if not _mamba_ok:
    print('FAILED — build log:')
    for line in log.strip().splitlines()[-60:]: print('    ' + line)
    raise RuntimeError('mamba-ssm install failed. Make sure GPU is enabled, then Runtime → Restart and run all.')

print('\nVerifying CUDA kernel...')
try:
    import selective_scan_cuda
    print('  selective_scan_cuda — OK')
except Exception as e:
    raise RuntimeError(f'selective_scan_cuda not importable: {e}\nTry: Runtime → Restart and run all')

import timm, einops
print(f'  timm   {timm.__version__} — OK')
print(f'  einops {einops.__version__} — OK')
print('\nAll dependencies ready.')

In [ ]:
# ================================================================
# CELL 3 — GET CODE FILES
# Clones your GitHub repo directly into /kaggle/working.
# Change GITHUB_REPO to your repo URL.
# ================================================================

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/FER-YOLO-Mamba.git'  # <-- change this
CLONE_DIR   = os.path.join(WORK_DIR, 'FER-YOLO-Mamba')

import subprocess

if os.path.exists(CLONE_DIR):
    print(f'Repo already cloned at {CLONE_DIR}, pulling latest...')
    r = subprocess.run(['git', 'pull'], cwd=CLONE_DIR, capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print(f'Cloning {GITHUB_REPO} ...')
    r = subprocess.run(
        ['git', 'clone', GITHUB_REPO, CLONE_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        raise RuntimeError(f'git clone failed:\n{r.stderr}')
    print('Clone done.')

# Set CODE_ROOT to the cloned directory
CODE_ROOT = CLONE_DIR

# RAF-DB is already mounted at the known path
MANUAL_DATASET_ROOT = '/kaggle/input/datasets/satyamsingh5512/dataset55121900'

# Walk into it to find the basic/ subfolder (EmoLabel/ + Annotation/ + Image/)
RAF_ROOT = None
for root, dirs, _ in os.walk(MANUAL_DATASET_ROOT):
    if all(m in dirs for m in ['EmoLabel', 'Annotation', 'Image']):
        RAF_ROOT = root
        break

if not RAF_ROOT:
    raise RuntimeError(
        f'EmoLabel/ + Annotation/ + Image/ not found inside {MANUAL_DATASET_ROOT}.\n'
        'Check that the RAF-DB dataset is mounted correctly.'
    )

print(f'Code root   : {CODE_ROOT}')
print(f'RAF-DB root : {RAF_ROOT}')

In [ ]:
# ================================================================
# CELL 4 — COPY CODE TO /kaggle/working
# Kaggle input is read-only. All outputs go to /kaggle/working.
# ================================================================
import shutil

print(f'Copying code → {WORK_DIR}')
for item in os.listdir(CODE_ROOT):
    src = os.path.join(CODE_ROOT, item)
    dst = os.path.join(WORK_DIR, item)
    if os.path.isdir(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

print(f'Working dir : {os.getcwd()}')
print(f'Contents    : {sorted(os.listdir("."))}')

In [ ]:
# ================================================================
# CELL 5 — GENERATE ANNOTATION FILES
# Converts RAF-DB → YOLO format. Safe to re-run.
# ================================================================
from convert_raf_to_yolo import convert

convert(RAF_ROOT, WORK_DIR)

print()
for fname in ['raf_train.txt', 'raf_val.txt']:
    lines = open(os.path.join(WORK_DIR, fname)).readlines()
    print(f'{fname}: {len(lines)} samples')
    print(f'  sample → {lines[0].strip()[:100]}')

In [ ]:
# ================================================================
# CELL 6 — PRE-FLIGHT CHECK
# All items must show OK before training starts.
# ================================================================
errors = []

def chk(label, ok, fix=''):
    status = 'OK  ' if ok else 'FAIL'
    print(f'  [{status}] {label}' + (f'  →  {fix}' if not ok and fix else ''))
    if not ok: errors.append(label)

print('Files:')
chk('nets/yolo.py',                os.path.isfile('nets/yolo.py'))
chk('nets/yolo_training.py',       os.path.isfile('nets/yolo_training.py'))
chk('utils/utils_fit.py',          os.path.isfile('utils/utils_fit.py'))
chk('train_kaggle.py',             os.path.isfile('train_kaggle.py'))
chk('convert_raf_to_yolo.py',      os.path.isfile('convert_raf_to_yolo.py'))
chk('model_data/sfew_classes.txt', os.path.isfile('model_data/sfew_classes.txt'))
chk('raf_train.txt',               os.path.isfile('raf_train.txt'), 'run Cell 5')
chk('raf_val.txt',                 os.path.isfile('raf_val.txt'),   'run Cell 5')

weights = 'model_data/yolox_s.pth'
if os.path.isfile(weights):
    print(f'  [OK  ] {weights}  ({os.path.getsize(weights)/1e6:.1f} MB)')
else:
    print(f'  [WARN] yolox_s.pth not found — training from scratch')

print('\nImports:')
try:
    import selective_scan_cuda; chk('selective_scan_cuda', True)
except Exception as e:
    chk('selective_scan_cuda', False, str(e))

try:
    from nets.yolo import YoloBody
    _ = YoloBody(7, 's'); del _
    chk('YoloBody(7, s) init', True)
except Exception as e:
    chk('YoloBody init', False, str(e))

if errors:
    raise RuntimeError('Pre-flight failed:\n  ' + '\n  '.join(errors))

print('\nAll checks passed.')

# Training config summary
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if   vram >= 30: fbs, ubs, inp = 32, 16, 640
elif vram >= 14: fbs, ubs, inp = 16,  8, 640
else:            fbs, ubs, inp =  8,  4, 512
print('\n=== Training config ===')
print(f'  GPU        : {torch.cuda.get_device_name(0)}  ({vram:.1f} GB)')
print(f'  Input      : {inp} x {inp}')
print(f'  Batch      : {fbs} frozen / {ubs} unfrozen')
print(f'  Epochs     : 300  (50 frozen + 250 unfrozen)')
print(f'  FP16       : True')
print(f'  Checkpoints: every 10 epochs → logs/')
print('=======================')

In [ ]:
# ================================================================
# CELL 7 — TRAIN
# Live output. Checkpoints save to logs/ every 10 epochs.
#   best_epoch_weights.pth  — best validation loss
#   last_epoch_weights.pth  — most recent epoch
# ================================================================
import subprocess

proc = subprocess.Popen(
    [sys.executable, '-u', 'train_kaggle.py'],
    cwd=WORK_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f'Training exited with code {proc.returncode}. Scroll up to see the error.')
print('\nTraining finished successfully.')

In [ ]:
# ================================================================
# CELL 8 — RESULTS
# Download from: Output tab → logs/
# ================================================================
import glob

checkpoints = sorted(glob.glob('logs/**/*.pth', recursive=True))
if not checkpoints:
    print('No checkpoints found yet.')
else:
    print(f'Saved checkpoints ({len(checkpoints)} total):')
    for ckpt in checkpoints:
        tag = '  <- BEST' if 'best' in ckpt else ('  <- LAST' if 'last' in ckpt else '')
        print(f'  {ckpt}  ({os.path.getsize(ckpt)/1e6:.1f} MB){tag}')

print('\nTo download: Output tab (right panel) → logs/')